# [3장 5강] - Multi-Head Attention Shape과 디버깅 (1)

<aside>
🎯

**실습 목표**

- Hidden 차원을 여러 head로 나누고 다시 합치는 shape 변환을 구현합니다.
- Multi-Head Self-Attention 전체 연산을 작성합니다.
- 선택 실습에서는 MHA와 GQA의 Query/KV head 수와 KV Cache 차이를 계산합니다.
- 잘못된 `view`·`transpose` 순서를 shape 계약으로 빠르게 찾습니다.
</aside>

---

## 핵심 실습. Head 분리와 병합 함수 작성

### 시작 코드

```python
import torch

torch.manual_seed(42)
X = torch.randn(2, 5, 12)  # [B, T, H]
num_heads = 3
```

### 수행해야 할 작업

1. `split_heads(X, num_heads)`를 작성해 `[B, heads, T, head_dim]`을 반환하세요.
2. `merge_heads(X_heads)`를 작성해 원래 `[B, T, H]`로 복원하세요.
3. Hidden size가 head 수로 나누어지지 않으면 오류를 발생시키세요.
4. 분리 후 병합한 값이 원본과 같은지 검증하세요.
    
  **해설**
    
  Head는 batch를 나누는 것이 아니라 hidden feature를 여러 부분 공간으로 나눕니다. `transpose(1, 2)`가 빠지면 head와 sequence 축의 의미가 뒤바뀝니다.

In [1]:
import torch

torch.manual_seed(42)
X = torch.randn(2, 5, 12)
num_heads = 3


def split_heads(x, num_heads):
    batch_size, seq_len, hidden_size = x.shape
    if hidden_size % num_heads != 0:
        raise ValueError("hidden_size는 num_heads로 나누어져야 합니다.")

    head_dim = hidden_size // num_heads
    # 먼저 hidden을 [heads, head_dim]으로 나눈 뒤 head 축을 앞으로 옮깁니다.
    return x.reshape(batch_size, seq_len, num_heads, head_dim).transpose(1, 2)


def merge_heads(x):
    batch_size, num_heads, seq_len, head_dim = x.shape
    # transpose 뒤에는 메모리 배치가 달라질 수 있어 contiguous 후 reshape합니다.
    return x.transpose(1, 2).contiguous().reshape(batch_size, seq_len, num_heads * head_dim)


X_heads = split_heads(X, num_heads)
X_restored = merge_heads(X_heads)
print("split:", tuple(X_heads.shape))
print("merged:", tuple(X_restored.shape))

assert X_heads.shape == (2, 3, 5, 4)
assert torch.allclose(X, X_restored)

split: (2, 3, 5, 4)
merged: (2, 5, 12)


## 핵심 보조 실습. Multi-Head Self-Attention 구현

### 시작 코드

```python
import torch
from torch import nn

torch.manual_seed(7)
X = torch.randn(2, 4, 8)
```

### 수행해야 할 작업

1. `MultiHeadSelfAttention(nn.Module)` 클래스를 작성하세요.
2. Q, K, V projection 후 head를 나누고 head별 attention을 계산하세요.
3. Head를 다시 합친 뒤 output projection을 적용하세요.
4. output `[2, 4, 8]`, weights `[2, 2, 4, 4]`를 반환하세요.
    
   **해설**
    
   Scaling에는 전체 hidden size가 아니라 head 하나의 차원인 `head_dim`을 사용합니다. 각 head가 독립적인 attention 분포를 만든 뒤 feature 방향으로 다시 합쳐집니다.

In [4]:
import math
import torch
from torch import nn

torch.manual_seed(7)
X = torch.randn(2, 4, 8)


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, hidden_size, num_heads):
        super().__init__()
        if hidden_size % num_heads != 0:
            raise ValueError("hidden_size는 num_heads로 나누어져야 합니다.")

        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.out_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def _split(self, x):
        B, T, _ = x.shape
        return x.reshape(B, T, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, x):
        Q = self._split(self.q_proj(x))
        K = self._split(self.k_proj(x))
        V = self._split(self.v_proj(x))

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        weights = torch.softmax(scores, dim=-1)
        context = torch.matmul(weights, V)

        B, _, T, _ = context.shape
        merged = context.transpose(1, 2).contiguous().reshape(B, T, self.hidden_size)
        return self.out_proj(merged), weights


model = MultiHeadSelfAttention(hidden_size=8, num_heads=2)
output, weights = model(X)
print(tuple(output.shape), tuple(weights.shape))


assert output.shape == (2, 4, 8)
assert weights.shape == (2, 2, 4, 4)

(2, 4, 8) (2, 2, 4, 4)


## 참고·심화 실습. GQA Config와 KV Shape 계산

### 시작 코드

```python
config = {
    "hidden_size": 4096,
    "num_attention_heads": 32,
    "num_key_value_heads": 8,
    "num_hidden_layers": 32,
}
batch_size = 2
sequence_length = 1024
bytes_per_value = 2
```

### 수행해야 할 작업

1. `head_dim`과 Query head당 K/V 공유 그룹 크기를 계산하세요.
2. Q와 K/V의 대표 shape을 반환하세요.
3. 전체 layer의 KV Cache 메모리를 MiB로 계산하세요.
4. 같은 설정의 MHA와 비교해 cache 감소 배수를 확인하세요.
    
   **해설**
    
  Query head 4개가 K/V head 하나를 공유합니다. Attention output의 Query head 수는 유지되지만 cache는 `H_kv`에 비례하므로 이 설정에서는 MHA보다 KV Cache가 4분의 1입니다.

In [6]:
config = {
    "hidden_size": 4096,
    "num_attention_heads": 32,
    "num_key_value_heads": 8,
    "num_hidden_layers": 32,
}
batch_size = 2
sequence_length = 1024
bytes_per_value = 2


def audit_gqa(config, batch_size, sequence_length, bytes_per_value=2):
    hidden_size = config["hidden_size"]
    h_q = config["num_attention_heads"]
    h_kv = config["num_key_value_heads"]
    layers = config["num_hidden_layers"]

    if hidden_size % h_q != 0:
        raise ValueError("hidden_size가 Query head 수로 나누어지지 않습니다.")
    if h_q % h_kv != 0:
        raise ValueError("Query head 수가 KV head 수로 나누어지지 않습니다.")

    head_dim = hidden_size // h_q
    group_size = h_q // h_kv
    cache_elements = 2 * layers * batch_size * sequence_length * h_kv * head_dim
    mha_elements = 2 * layers * batch_size * sequence_length * h_q * head_dim

    return {
        "head_dim": head_dim,
        "query_heads_per_kv_head": group_size,
        "q_shape": (batch_size, h_q, sequence_length, head_dim),
        "kv_shape": (batch_size, h_kv, sequence_length, head_dim),
        "gqa_cache_mib": cache_elements * bytes_per_value / (1024 ** 2),
        "mha_cache_mib": mha_elements * bytes_per_value / (1024 ** 2),
        "reduction_ratio": mha_elements / cache_elements,
    }


report = audit_gqa(config, batch_size, sequence_length, bytes_per_value)
print(report)

assert report["head_dim"] == 128
assert report["query_heads_per_kv_head"] == 4
assert report["reduction_ratio"] == 4

{'head_dim': 128, 'query_heads_per_kv_head': 4, 'q_shape': (2, 32, 1024, 128), 'kv_shape': (2, 8, 1024, 128), 'gqa_cache_mib': 256.0, 'mha_cache_mib': 1024.0, 'reduction_ratio': 4.0}


## 참고·심화 실습. Shape Trace 디버거 작성

### 시작 코드

```python
trace_good = {
    "input": (2, 5, 12),
    "qkv": (2, 5, 12),
    "split": (2, 3, 5, 4),
    "scores": (2, 3, 5, 5),
    "merged": (2, 5, 12),
}

trace_bad = {
    "input": (2, 5, 12),
    "qkv": (2, 5, 12),
    "split": (2, 5, 3, 4),
    "scores": (2, 5, 3, 3),
    "merged": (2, 5, 12),
}
```

### 수행해야 할 작업

1. `audit_mha_trace(trace, num_heads)` 함수를 작성하세요.
2. Input에서 B, T, H를 읽고 각 단계의 기대 shape를 계산하세요.
3. 처음 어긋난 단계, 기대 shape, 실제 shape를 반환하세요.
4. 정상 trace에는 `valid=True`를 반환하세요.

    
  **해설**
    
  오류 메시지가 마지막 matmul에서 발생하더라도 원인은 그 앞의 head 분리일 수 있습니다. Shape trace는 첫 번째 계약 위반 지점을 찾는 데 목적이 있습니다.

In [7]:
# Multi-Head Attention(MHA)의 각 단계에서 텐서 shape가
# 올바르게 변했는지 검사하는 검증기(audit)

trace_good = {
    "input": (2, 5, 12), "qkv": (2, 5, 12), "split": (2, 3, 5, 4),
    "scores": (2, 3, 5, 5), "merged": (2, 5, 12),
}
trace_bad = {
    "input": (2, 5, 12), "qkv": (2, 5, 12), "split": (2, 5, 3, 4),
    "scores": (2, 5, 3, 3), "merged": (2, 5, 12),
}


def audit_mha_trace(trace, num_heads):
    B, T, H = trace["input"]
    if H % num_heads != 0:
        return {"valid": False, "stage": "config", "reason": "hidden_not_divisible"}

    D = H // num_heads
    expected = {
        "qkv": (B, T, H),
        "split": (B, num_heads, T, D),
        "scores": (B, num_heads, T, T),
        "merged": (B, T, H),
    }

    for stage, expected_shape in expected.items():
        if trace.get(stage) != expected_shape:
            return {
                "valid": False,
                "stage": stage,
                "expected": expected_shape,
                "actual": trace.get(stage),
            }
    return {"valid": True}


print(audit_mha_trace(trace_good, 3))
print(audit_mha_trace(trace_bad, 3))
assert audit_mha_trace(trace_good, 3)["valid"]
assert audit_mha_trace(trace_bad, 3)["stage"] == "split"

{'valid': True}
{'valid': False, 'stage': 'split', 'expected': (2, 3, 5, 4), 'actual': (2, 5, 3, 4)}
